In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## pre processing

In [8]:
# lower case
df["review"]=df["review"].str.lower()

In [9]:
# removing url
import re
def remove_url(text):
    text=re.sub(r"http\S+","",text)
    return text
df["review"]=df["review"].apply(remove_url)

In [10]:
# remove punctuations

def punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(punctuations)

In [11]:
# remove html

def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

In [12]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    return " ".join(tokens)

df["review"] = df["review"].apply(remove_stopwords)

In [14]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [15]:
# stemming
from nltk.stem import PorterStemmer

In [16]:
ps = PorterStemmer()

def stemming(text):
    tokens = word_tokenize(text)
    stemmed_words = [ps.stem(token) for token in tokens]
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [17]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [18]:
# encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [21]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [22]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108221 stored elements and shape (49582, 5000)>

## Dataset and DataLoader

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [25]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [26]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values.copy()).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values.copy()).float()
)

In [27]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=False, batch_size=64)

# build RNN

In [28]:
import torch.nn as nn
import torch.optim as optim

In [29]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [30]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [31]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        loss = criterion(outputs.squeeze(), yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * Xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"epoch = {epoch+1}/{epochs} and loss = {epoch_loss:.4f}")

epoch = 1/10 and loss = 0.3440
epoch = 2/10 and loss = 0.2327
epoch = 3/10 and loss = 0.2166
epoch = 4/10 and loss = 0.2102
epoch = 5/10 and loss = 0.2055
epoch = 6/10 and loss = 0.2023
epoch = 7/10 and loss = 0.2002
epoch = 8/10 and loss = 0.1979
epoch = 9/10 and loss = 0.1966
epoch = 10/10 and loss = 0.1950


In [33]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.31471211051729


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim

In [35]:
class DeepMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

In [36]:
input_size = X_train.shape[1] # 5000 features
model = DeepMLP(input_size)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [37]:
epochs = 12

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(Xb)  # Accepts 2D tensor (batch_size, 5000)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * Xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

Epoch 1/12 - Loss: 0.3237
Epoch 2/12 - Loss: 0.2127
Epoch 3/12 - Loss: 0.1497
Epoch 4/12 - Loss: 0.1020
Epoch 5/12 - Loss: 0.0755
Epoch 6/12 - Loss: 0.0595
Epoch 7/12 - Loss: 0.0483
Epoch 8/12 - Loss: 0.0398
Epoch 9/12 - Loss: 0.0371
Epoch 10/12 - Loss: 0.0307
Epoch 11/12 - Loss: 0.0275
Epoch 12/12 - Loss: 0.0237


In [38]:
model.eval()
correct_vals = 0
tot_vals = 0

with torch.no_grad():
    for Xb, yb in test_loader:
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"Accuracy = {(correct_vals / tot_vals) * 100:.2f}%")

Accuracy = 87.59%


In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Train / Test Split using your preprocessed df['review']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["review"], df["sentiment"], test_size=0.2, random_state=42
)

# 2. High-capacity TF-IDF with Unigrams, Bigrams, and Trigrams
tfidf = TfidfVectorizer(
    max_features=50000,        
    ngram_range=(1, 3),         
    sublinear_tf=True           
)

X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

# 3. Logistic Regression with strong regularization scaling
model = LogisticRegression(C=5.0, max_iter=1000, solver="saga")
model.fit(X_train, y_train)

# 4. Predict & Evaluate
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Upgraded TF-IDF Model Accuracy: {acc * 100:.2f}%")

Upgraded TF-IDF Model Accuracy: 90.16%
